# Modify · Transformación

Se limpia el corpus y se normaliza su huella léxica para eliminar duplicados. El pipeline
tokeniza palabras y signos, agrega bigramas y observaciones estáticas, aplica TF-IDF
logarítmico y normaliza los vectores con norma L2. La regresión logística usa pesos
balanceados para cada problema binario.

In [1]:
import json
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "pyproject.toml").is_file():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
results = root / "results"
def read(name):
    return json.loads((results / name).read_text(encoding="utf-8"))

In [2]:
from lab.sample import sample
from vulnminer.features import terms
from vulnminer.model import fit_vocabulary, vectorize
rows, _ = sample()
training = [row for row in rows if row["split"] == "train"]
vocabulary = fit_vocabulary([terms(row["text"], row["language"]) for row in training])
row = rows[0]
transformed = vectorize(terms(row["text"], row["language"]), vocabulary)
print("Filas:", len(rows), "Columnas:", len(vocabulary))
print("Primera fila de la matriz dispersa:", row["id"], row["split"], sorted(transformed.items())[:8])
print("Observaciones en el vocabulario:", [term for term in vocabulary if term.startswith("signal:")])

Filas: 868 Columnas: 628
Primera fila de la matriz dispersa: VM-2c5bccb0982a train [(4, 0.05272078577849797), (26, 0.09499899730977114), (65, 0.0741347787410402), (82, 0.17541046247537673), (92, 0.08149303847335373), (127, 0.14183211522704822), (131, 0.2178955016097936), (319, 0.17541046247537673)]
Observaciones en el vocabulario: ['signal:debug', 'signal:debug_disabled', 'signal:debug_enabled', 'signal:error_action', 'signal:handler', 'signal:ignored_handler', 'signal:lookup', 'signal:owner_scope', 'signal:parameter_binding', 'signal:quantity', 'signal:quantity_bound', 'signal:request_input', 'signal:sql', 'signal:sql_composition', 'signal:token', 'signal:token_decode', 'signal:token_verify', 'signal:transport', 'signal:transport_disabled', 'signal:transport_enabled', 'signal:unbounded']


El vocabulario y las ponderaciones IDF se ajustan con entrenamiento y se reutilizan en validación, prueba e inferencia.